# Class-balanced focal loss benchmark
Train and evaluate each backbone sequentially. Results from all three notebooks are collected in one CSV file.

In [ ]:
import csv
import gc
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score

from modules.dataset import build_data_pipelines
from modules.foundation import (
    RETFoundDINO, RETFoundMAE, DINOv2Classifier,
    EyeCLIPClassifier, BioMedCLIPClassifier,
)
from modules.losses import ClassBalancedFocalLoss, SupConLoss

In [ ]:
# Update these paths and settings for your machine.
DATA_DIR = "data/AHEH"
WEIGHTS_DIR = Path("weights")
RESULTS_DIR = Path("results/notebook_benchmark")
CSV_PATH = RESULTS_DIR / "all_model_results.csv"
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 50
PATIENCE = 7
UNFREEZE_BLOCKS = 4
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 5e-2
PROJECTION_DIM = 128
SUPCON_WEIGHT = 0.2
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

In [ ]:
# The repository split is created once so every model uses the same train/validation/test images.
pipelines = build_data_pipelines(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    train_size=0.7,
    seed=SEED,
    p=8,
    k=2,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
train_loader = pipelines["train_loader"]
contrastive_loader = pipelines["train_loader_contrastive"]
val_loader = pipelines["val_loader"]
test_loader = pipelines["test_loader"]
class_names = pipelines["class_names"]
train_labels = pipelines["train_labels"]
num_classes = pipelines["num_classes"]
class_counts = np.bincount(train_labels, minlength=num_classes).tolist()
print(f"{num_classes} classes; train counts:", class_counts)

In [ ]:
# Each factory constructs one model at a time. Edit checkpoint paths to match your downloaded weights.
MODEL_FACTORIES = {
    "RETFound_DINOv2": lambda: RETFoundDINO(
        num_classes=num_classes, projection_dim=PROJECTION_DIM,
        checkpoint_path=str(WEIGHTS_DIR / "RETFound_dinov2_shanghai.pth"),
        image_size=IMAGE_SIZE,
    ),
    "RETFound_MAE": lambda: RETFoundMAE(
        num_classes=num_classes, projection_dim=PROJECTION_DIM,
        checkpoint_path=str(WEIGHTS_DIR / "RETFound_mae_shanghai.pth"),
        image_size=IMAGE_SIZE,
    ),
    "DINOv2": lambda: DINOv2Classifier(
        num_classes=num_classes, projection_dim=PROJECTION_DIM,
        model_name="dinov2_vitl14", pretrained=True,
    ),
    "EyeCLIP": lambda: EyeCLIPClassifier(
        num_classes=num_classes, projection_dim=PROJECTION_DIM,
        checkpoint_path=str(WEIGHTS_DIR / "eyeclip_visual.pt"),
        image_size=IMAGE_SIZE,
    ),
    "BioMedCLIP": lambda: BioMedCLIPClassifier(
        num_classes=num_classes, projection_dim=PROJECTION_DIM, pretrained=True,
    ),
}

# EyeCLIP and BioMedCLIP use CLIP-style normalization; the dataset pipeline emits ImageNet-normalized tensors.
CLIP_MEAN = torch.tensor((0.48145466, 0.4578275, 0.40821073), device=DEVICE).view(1, 3, 1, 1)
CLIP_STD = torch.tensor((0.26862954, 0.26130258, 0.27577711), device=DEVICE).view(1, 3, 1, 1)
IMAGENET_MEAN = torch.tensor((0.485, 0.456, 0.406), device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor((0.229, 0.224, 0.225), device=DEVICE).view(1, 3, 1, 1)

def prepare_images(images, clip_normalization):
    images = images.to(DEVICE, non_blocking=True)
    if clip_normalization:
        images = (images * IMAGENET_STD + IMAGENET_MEAN - CLIP_MEAN) / CLIP_STD
    return images

In [ ]:
def append_result(row):
    """Upsert one model/method result in the shared CSV."""
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    old_rows = []
    if CSV_PATH.exists():
        with CSV_PATH.open(newline="") as handle:
            old_rows = list(csv.DictReader(handle))
    old_rows = [r for r in old_rows if not (r["model"] == row["model"] and r["method"] == row["method"])]
    old_rows.append({key: str(value) for key, value in row.items()})
    columns = list(row)
    with CSV_PATH.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(old_rows)

def train_evaluate_one(model_name, factory, method):
    clip_normalization = model_name in {"EyeCLIP", "BioMedCLIP"}
    started = time.time()
    model = factory().to(DEVICE)
    model.unfreeze_last_n_blocks(UNFREEZE_BLOCKS)
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    ce_loss = nn.CrossEntropyLoss()
    focal_loss = ClassBalancedFocalLoss(class_counts).to(DEVICE)
    supcon_loss = SupConLoss(temperature=0.07)
    loader = contrastive_loader if method == "Focal + SupCon" else train_loader
    checkpoint = RESULTS_DIR / f"{model_name}_{method.replace(' ', '_').replace('+', 'plus')}.pth"
    best_f1, stale_epochs = -1.0, 0
    use_bf16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()

    try:
        for epoch in range(EPOCHS):
            model.train()
            for images, labels in loader:
                images = prepare_images(images, clip_normalization)
                labels = labels.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=use_bf16):
                    if method == "Focal + SupCon":
                        logits, projections = model(images, mode="both")
                        loss = focal_loss(logits, labels) + SUPCON_WEIGHT * supcon_loss(projections, labels)
                    else:
                        logits = model(images, mode="classification")
                        loss = ce_loss(logits, labels) if method == "Cross Entropy" else focal_loss(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
                optimizer.step()

            model.eval()
            val_preds, val_true = [], []
            with torch.inference_mode():
                for images, labels in val_loader:
                    images = prepare_images(images, clip_normalization)
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=use_bf16):
                        logits = model(images, mode="classification")
                    val_preds.extend(logits.argmax(1).cpu().tolist())
                    val_true.extend(labels.tolist())
            val_f1 = f1_score(val_true, val_preds, average="macro", zero_division=0)
            print(f"{model_name} | {method} | epoch {epoch + 1}/{EPOCHS} | val macro-F1={val_f1:.4f}")
            if val_f1 > best_f1:
                best_f1, stale_epochs = val_f1, 0
                torch.save(model.state_dict(), checkpoint)
            else:
                stale_epochs += 1
                if stale_epochs >= PATIENCE:
                    break

        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
        model.eval()
        test_preds, test_true = [], []
        with torch.inference_mode():
            for images, labels in test_loader:
                images = prepare_images(images, clip_normalization)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=use_bf16):
                    logits = model(images, mode="classification")
                test_preds.extend(logits.argmax(1).cpu().tolist())
                test_true.extend(labels.tolist())

        result = {
            "model": model_name,
            "method": method,
            "accuracy": accuracy_score(test_true, test_preds),
            "balanced_accuracy": balanced_accuracy_score(test_true, test_preds),
            "macro_precision": precision_score(test_true, test_preds, average="macro", zero_division=0),
            "macro_recall": recall_score(test_true, test_preds, average="macro", zero_division=0),
            "macro_f1": f1_score(test_true, test_preds, average="macro", zero_division=0),
            "weighted_f1": f1_score(test_true, test_preds, average="weighted", zero_division=0),
            "test_images": len(test_true),
            "checkpoint": str(checkpoint),
            "seconds": round(time.time() - started, 1),
        }
        append_result(result)
        return result
    finally:
        del model, optimizer, ce_loss, focal_loss, supcon_loss
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("GPU cache cleared.")

In [ ]:
# Runs all five backbones in sequence and updates the shared CSV after each completed evaluation.
METHOD = "Focal Loss"
results = []
for name, factory in MODEL_FACTORIES.items():
    print(f"\n=== {name} | {METHOD} ===")
    row = train_evaluate_one(name, factory, METHOD)
    results.append(row)
    print(row)

print(f"\nResults saved to {CSV_PATH}")
